# Ensemble Affinity Analysis (On-the-fly)

This notebook computes PRP and RA summaries in memory from merged benchmark files.

No summary CSVs are created and figures are shown inline only.

In [9]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px

In [10]:
# Set run_id explicitly, or keep None to use latest run folder.
ignore_models = ['smollm2-1.7b', 'gemma3-4b', 'phi3-3.8b']
run_id =None
# run_id = "20260401_082538"
# run_id = "20260406_211752"

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
bench_root = repo_root / "data" / "results" / "affinity_benchmark"

if run_id is None:
    run_dirs = sorted([p for p in bench_root.iterdir() if p.is_dir()])
    if not run_dirs:
        raise FileNotFoundError(f"No benchmark runs found under {bench_root}")
    run_dir = run_dirs[-1]
else:
    run_dir = bench_root / run_id

prp_path = run_dir / "merged_prp_affinities.csv"
ra_path = run_dir / "merged_ra_affinities.csv"

if not prp_path.exists() :
    raise FileNotFoundError(
        "Merged files not found. Run scripts/run_postprocessing.py first for this run_id."
    )
prp_df = pd.read_csv(prp_path)
prp_df = prp_df[~prp_df['Model_Slug'].isin(ignore_models)]

print("Run directory:", run_dir)
print("PRP rows:", len(prp_df), "| Models:", prp_df['Model_Slug'].nunique())
if ra_path.exists():
    ra_df = pd.read_csv(ra_path)
    ra_df = ra_df[~ra_df['Model_Slug'].isin(ignore_models)]
    print("RA rows:", len(ra_df), "| Models:", ra_df['Model_Slug'].nunique())

Run directory: /mnt/c/Users/pgallego/OneDrive - Imperial College London/PhD/Repositories/article-classifier/data/results/affinity_benchmark/20260413_151017
PRP rows: 3000 | Models: 5


In [11]:
# Quick coverage checks
display(prp_df[['Model_Slug', 'Abstract_Index']].drop_duplicates().groupby('Model_Slug').size().rename('n_abstracts').reset_index())
display(ra_df[['Model_Slug', 'Abstract_Index']].drop_duplicates().groupby('Model_Slug').size().rename('n_abstracts').reset_index())

,Model_Slug,n_abstracts
0,gemini-3-flash-preview,100
1,gemma4-31b-cloud,100
2,gpt-5.4-mini,100
3,gpt-5.4-nano,100
4,ministral-3-14b-cloud,100


NameError: name 'ra_df' is not defined

# PRP Affinity level
Affinity level of an **abstract** to each **primary research programme** (PRP)

In [12]:
fig = px.box(
    prp_df,
    x='Abstract_Index',
    y='LLM_Affinity',
    points='all',
    color = 'PRP_Name',
    title=f'PRP Affinity Distribution by Abstract'
 )
fig.update_xaxes(range=[5.5,11.5])
fig.update_layout(xaxis_title='Abstract Index', yaxis_title='LLM Affinity')
fig.show()

In [13]:
fig = px.box(
    prp_df,
    color='Model_Slug',
    y='LLM_Affinity',
    x = 'PRP_Name',
    # points = 'all',
    title='Affinity Distribution by PRP across models',
 )
# fig.update_layout(xaxis_title='PRP', yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

In [14]:
# fig = px.box(
#     prp_df,
#     x='Abstract_Index',
#     y='LLM_Affinity',
#     points='all',
#     color = 'PRP_Name',
#     title=f'PRP Affinity Distribution by Abstract'
#  )
# fig.update_xaxes(range=[0.5,10.5])
# fig.update_layout(xaxis_title='Abstract Index', yaxis_title='LLM Affinity')
# fig.show()

In [15]:
abstracts_ids = [1,8,20, 19]
abstracts_ids = [20, 25, 34]
fig = px.box(
    prp_df[prp_df['Abstract_Index'].isin(abstracts_ids)], #ra_df[ra_df['Abstract_Index'].isin([7,8,9])],
    # x='Abstract_Index',
    y='LLM_Affinity',
    # color='Model_Slug',
    points='all',
    # hover_data='PRP_Name',
    # facet_row='PRP_Name',
    facet_col='Abstract_Index',
    color = 'PRP_Name',
    # x = 'Model_Slug',
    title=f'PRP Affinity Distribution by Abstract'
 )
# fig.update_xaxes(range=[0.5,10.5])
fig.update_layout(xaxis_title='Abstract Index', yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

In [16]:
fig = px.line(
    prp_df[prp_df['Abstract_Index'].isin([1,8,20, 19, 30])], #ra_df[ra_df['Abstract_Index'].isin([7,8,9])],
    # x='Abstract_Index',
    y='LLM_Affinity',
    # color='Model_Slug',
    # points='all',
    # hover_data='PRP_Name',
    # facet_row='PRP_Name',

    facet_col='Abstract_Index',
    color = 'PRP_Name',
    x = 'Model_Slug',
    title=f'PRP Affinity Distribution by Abstract',
    markers = True
 )
# fig.update_xaxes(range=[0.5,10.5])
fig.update_layout(yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

General categories:
- 1 - Scenario Reduction Network Based on Wasserstein Distance With Regularization - Planning
- 8 - Simultaneous Assessment of Multiple Aspects of Stability of Power Systems With Renewable Generation -  Stability
- 19 - Chance-Constrained Pre-Contingency Joint Self- Scheduling of Energy and Reserve in VPP - DER, Services
- 20 - Dual Grid-Forming Control With Energy Regulation Capability of MMC-HVDC System Integrating Offshore Wind Farms and Weak Grids - IBR, Stability
- 30 - Synergistic Integration of Machine Learning and Mathematical Optimization for Unit Commitment - ????

Observations:
- Gemma: fails to identify planning for Index 1
- Gemma cloud: best classification
- Llama: miscelaneous performance
- Mistral: systematically assign too high affinities
- Phi & Smollm: systematically assign too high affinities
- Qwen: miscelaneous


## PRP Summaries

In [17]:
prp_summary = (
    prp_df.groupby(['Model_Slug', 'PRP_Name'], as_index=False)['LLM_Affinity']
    .agg(mean_affinity='mean', std_affinity='std', n='count')
)
display(prp_summary.head(20))

,Model_Slug,PRP_Name,mean_affinity,std_affinity,n
0,gemini-3-flash-preview,CROF,35.34,25.095865,100
1,gemini-3-flash-preview,DER,34.48,30.271659,100
2,gemini-3-flash-preview,IBR,33.71,28.320780,100
3,gemini-3-flash-preview,Planning,50.75,27.327559,100
4,gemini-3-flash-preview,Stability,41.17,33.647326,100
5,gemini-3-flash-preview,System Services,35.56,22.290697,100
6,gemma4-31b-cloud,CROF,35.85,24.659858,100
7,gemma4-31b-cloud,DER,35.65,29.376131,100
8,gemma4-31b-cloud,IBR,40.88,26.909230,100
9,gemma4-31b-cloud,Planning,48.80,27.717186,100


In [18]:
# 
fig = px.scatter(
    prp_summary,
    color='PRP_Name',
    y='mean_affinity',
    x = 'Model_Slug',
    title='PRP Affinity Distribution by Model',
    # points = 'all',
 )
fig.update_layout(xaxis_title='PRP', yaxis_title='LLM Affinity')
fig.update_xaxes(tickangle=45)
fig.show()

In [19]:
prp_heat = prp_summary.pivot(index='Model_Slug', columns='PRP_Name', values='mean_affinity').sort_index()
fig = px.imshow(
    prp_heat,
    text_auto='.1f',
    color_continuous_scale='Viridis',
    title='Mean PRP Affinity (Model x PRP)',
    aspect='auto'
 )
fig.update_layout(xaxis_title='PRP Name', yaxis_title='Model')
fig.show()

In [20]:
# Top-1 PRP agreement per abstract
prp_top1 = (
    prp_df.sort_values(['Model_Slug', 'Abstract_Index', 'LLM_Affinity'], ascending=[True, True, False])
    .groupby(['Model_Slug', 'Abstract_Index'], as_index=False)
    .first()[['Model_Slug', 'Abstract_Index', 'PRP_Name','LLM_Affinity']]
)

prp_agreement = (
    prp_top1.groupby('Abstract_Index')['PRP_Name']
    .nunique()
    .rename('n_unique_top1_prp')
    .reset_index()
)

print('Full agreement rate (all models same top-1 PRP):', (prp_agreement['n_unique_top1_prp'] == 1).mean())
display(prp_agreement['n_unique_top1_prp'].value_counts().sort_index().rename('n_abstracts'))

Full agreement rate (all models same top-1 PRP): 0.53


n_unique_top1_prp
1    53
2    33
3    13
4     1
Name: n_abstracts, dtype: int64

## RA Summaries

In [21]:
ra_df.head(20)

NameError: name 'ra_df' is not defined

In [ ]:
# fig = px.box(
#     ra_df,
#     x='Abstract_Index',
#     y='LLM_Affinity',
#     color='Model_Slug',
#     points='all',
#     # facet_row='PRP_Name',
#     # color = 'PRP_Name',
#     title=f'RA Affinity Distribution by Abstract'
#  )
# fig.update_layout(xaxis_title='Abstract Index', yaxis_title='LLM Affinity')
# # fig.update_xaxes(tickangle=45)
# fig.show()

In [ ]:
fig = px.scatter(#prp_df[prp_df['Abstract_Index'].isin([1,8,20,19])]
    ra_df[ra_df['Abstract_Index'].isin([1,8,20,19])],
    # ra_df,
    x='RA2025_ID',
    y='LLM_Affinity',
    color='Model_Slug',
    facet_col = 'Abstract_Index',
    # points='all',
    hover_data = 'Model_Slug',
    # facet_row='PRP_Name',
    # color = 'PRP_Name',
    # title=f'RA Affinity Distribution by Abstract'
 )
fig.update_layout(xaxis_title='RA question', yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
ra_summary = (
    ra_df.groupby(['Model_Slug', 'RA2025_ID'], as_index=False)['LLM_Affinity']
    .agg(mean_affinity='mean', std_affinity='std', n='count')
)
display(ra_summary.head(20))

,Model_Slug,RA2025_ID,mean_affinity,std_affinity,n
0,gemma4-31b-cloud,1,41.875000,27.894892,8
1,gemma4-31b-cloud,2,28.750000,44.417527,4
2,gemma4-31b-cloud,3,60.000000,40.207794,4
3,gemma4-31b-cloud,4,63.750000,42.695628,4
4,gemma4-31b-cloud,5,45.000000,36.968455,4
5,gemma4-31b-cloud,6,11.250000,8.539126,4
6,gemma4-31b-cloud,7,63.888889,19.965248,9
7,gemma4-31b-cloud,8,40.555556,15.092309,9
8,gemma4-31b-cloud,9,79.444444,4.639804,9
9,gemma4-31b-cloud,10,33.333333,5.000000,9


In [22]:
fig = px.box(
    prp_df,
    x='Model_Slug',
    y='LLM_Affinity',
    title='RA Affinity Distribution by Model'
 )
fig.update_layout(xaxis_title='Model', yaxis_title='LLM Affinity')
fig.update_xaxes(tickangle=45)
fig.show()

In [23]:
fig = px.box(
    ra_df,
    x='Model_Slug',
    y='LLM_Affinity',
    title='RA Affinity Distribution by Model'
 )
fig.update_layout(xaxis_title='Model', yaxis_title='LLM Affinity')
fig.update_xaxes(tickangle=45)
fig.show()

NameError: name 'ra_df' is not defined

In [ ]:
# Top-1 RA agreement per abstract
ra_top1 = (
    ra_df.sort_values(['Model_Slug', 'Abstract_Index', 'LLM_Affinity'], ascending=[True, True, False])
    .groupby(['Model_Slug', 'Abstract_Index'], as_index=False)
    .first()[['Model_Slug', 'Abstract_Index', 'RA2025_ID']]
)

ra_agreement = (
    ra_top1.groupby('Abstract_Index')['RA2025_ID']
    .nunique()
    .rename('n_unique_top1_ra')
    .reset_index()
)

print('Full agreement rate (all models same top-1 RA):', (ra_agreement['n_unique_top1_ra'] == 1).mean())
display(ra_agreement['n_unique_top1_ra'].value_counts().sort_index().rename('n_abstracts'))

Full agreement rate (all models same top-1 RA): 0.0


n_unique_top1_ra
2     1
3     5
4    10
5    14
6    17
7     3
Name: n_abstracts, dtype: int64

## Coherency Analysis: Model Agreement on Affinity Levels

Analyze whether all models assign similar affinity scores to the same questions/abstracts.

## Coherency Logic: Coefficient of Variation (CV)

The coefficient of variation measures how dispersed the model affinities are relative to their average value.  
For each abstract-target unit, we collect the affinity scores from all models and compute:

$$
\mu = \frac{1}{n}\sum_{i=1}^{n} x_i
$$

$$
\sigma = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\mu)^2}
$$

$$
CV = \frac{\sigma}{\mu + \varepsilon}
$$

where:

- $x_i$ is the affinity score assigned by model $i$
- $n$ is the number of models
- $\mu$ is the mean affinity
- $\sigma$ is the sample standard deviation
- $\varepsilon$ is a small constant added to avoid division by zero

Interpretation:

- Low CV means the models are in close agreement
- High CV means the models disagree more strongly
- CV can become unstable when the mean affinity is very small, so low-mean cases should be interpreted with caution

In [24]:
def affinity_level_fixed(score):
    if pd.isna(score):
        return pd.NA
    try:
        x = float(score)
    except Exception:
        return pd.NA
    if 0 <= x <= 40:
        return 'Low'
    if 40 < x <= 70:
        return 'Moderate'
    if 70 < x <= 100:
        return 'High'
    return pd.NA


def compute_unit_coherency(df, target_col, eps=1e-8, low_mean_threshold=5.0):
    required_cols = {'Abstract_Index', 'Model_Slug', target_col, 'LLM_Affinity'}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    unit_stats = (
        df.groupby(['Abstract_Index', target_col], as_index=False)['LLM_Affinity']
        .agg(
            mean_affinity='mean',
            std_affinity='std',
            min_affinity='min',
            max_affinity='max',
            n_models='count',
            mad_affinity=lambda s: float(np.mean(np.abs(s - s.mean()))),
        )
    )
    unit_stats['cv'] = unit_stats['std_affinity'] / (unit_stats['mean_affinity'] + eps)
    unit_stats.loc[unit_stats['n_models'] < 2, 'cv'] = np.nan
    unit_stats['range'] = unit_stats['max_affinity'] - unit_stats['min_affinity']
    unit_stats['Affinity_Level'] = unit_stats['mean_affinity'].apply(affinity_level_fixed)
    unit_stats['cv_low_mean_flag'] = unit_stats['mean_affinity'] < float(low_mean_threshold)
    unit_stats['singleton_flag'] = unit_stats['n_models'] < 2
    return unit_stats


def summarize_coherency_by_level(unit_stats):
    level_order = pd.CategoricalDtype(categories=['Low', 'Moderate', 'High'], ordered=True)
    out = unit_stats.copy()
    out['Affinity_Level'] = out['Affinity_Level'].astype(level_order)

    level_summary = (
        out.dropna(subset=['Affinity_Level'])
        .groupby('Affinity_Level', as_index=False, observed=False)
        .agg(
            n_units=('Affinity_Level', 'count'),
            mean_cv=('cv', 'mean'),
            median_cv=('cv', 'median'),
            p90_cv=('cv', lambda s: float(np.percentile(s.dropna(), 90)) if s.notna().any() else np.nan),
            mean_mad=('mad_affinity', 'mean'),
            median_mad=('mad_affinity', 'median'),
            mean_range=('range', 'mean'),
            pct_low_mean_flag=('cv_low_mean_flag', 'mean'),
            pct_singleton=('singleton_flag', 'mean'),
        )
        .sort_values('Affinity_Level')
    )
    level_summary['pct_low_mean_flag'] = 100.0 * level_summary['pct_low_mean_flag']
    level_summary['pct_singleton'] = 100.0 * level_summary['pct_singleton']
    return level_summary


def compute_pairwise_disagreement_by_level(df, target_col):
    records = []
    for (abstract_idx, target_id), grp in df.groupby(['Abstract_Index', target_col]):
        grp2 = grp[['Model_Slug', 'LLM_Affinity']].dropna()
        if grp2['Model_Slug'].nunique() < 2:
            continue

        level = affinity_level_fixed(grp2['LLM_Affinity'].mean())
        rows = grp2.sort_values('Model_Slug')[['Model_Slug', 'LLM_Affinity']].to_records(index=False)

        for i in range(len(rows)):
            for j in range(i + 1, len(rows)):
                m1, a1 = rows[i]
                m2, a2 = rows[j]
                records.append(
                    {
                        'Abstract_Index': abstract_idx,
                        'Target_ID': target_id,
                        'Affinity_Level': level,
                        'Model_1': str(m1),
                        'Model_2': str(m2),
                        'Affinity_Difference': abs(float(a1) - float(a2)),
                    }
                )

    pairwise = pd.DataFrame(records)
    if pairwise.empty:
        return pairwise, pairwise

    pairwise_summary = (
        pairwise.dropna(subset=['Affinity_Level'])
        .groupby(['Affinity_Level', 'Model_1', 'Model_2'], as_index=False)
        .agg(mean_diff=('Affinity_Difference', 'mean'), median_diff=('Affinity_Difference', 'median'), std_diff=('Affinity_Difference', 'std'), n_pairs=('Affinity_Difference', 'count'))
        .sort_values(['Affinity_Level', 'mean_diff'], ascending=[True, False])
    )
    return pairwise, pairwise_summary

In [25]:
prp_coherency = compute_unit_coherency(prp_df, target_col='PRP_Name')

print("PRP Coherency Summary:")
print(f"Mean CV across all abstract-PRP pairs: {prp_coherency['cv'].mean():.3f}")
print(f"Abstracts with perfect agreement (CV=0): {(prp_coherency['cv'] == 0).sum()}")
print("\nBottom 10 coherent (high CV) abstract-PRP pairs:")
display(prp_coherency.nlargest(10, 'cv')[['Abstract_Index', 'PRP_Name', 'mean_affinity', 'cv', 'range']])
print("\nTop 10 coherent (low CV) abstract-PRP pairs:")
display(prp_coherency.nsmallest(10, 'cv')[['Abstract_Index', 'PRP_Name', 'mean_affinity', 'cv', 'range']])

PRP Coherency Summary:
Mean CV across all abstract-PRP pairs: 0.530
Abstracts with perfect agreement (CV=0): 0

Bottom 10 coherent (high CV) abstract-PRP pairs:


,Abstract_Index,PRP_Name,mean_affinity,cv,range
318,54,CROF,2.0,2.236068,10.0
13,3,DER,4.0,1.629801,15.0
199,34,DER,3.0,1.490712,10.0
202,34,Stability,3.0,1.490712,10.0
475,80,DER,3.0,1.490712,10.0
474,80,CROF,10.0,1.414214,30.0
205,35,DER,5.0,1.414214,15.0
479,80,System Services,7.0,1.392399,20.0
17,3,System Services,8.0,1.369306,20.0
336,57,CROF,6.0,1.369306,20.0



Top 10 coherent (low CV) abstract-PRP pairs:


,Abstract_Index,PRP_Name,mean_affinity,cv,range
262,44,Stability,90.4,0.009894,2.0
116,20,IBR,96.4,0.015732,3.0
123,21,Planning,94.6,0.016031,4.0
322,54,Stability,96.2,0.017081,3.0
454,76,Stability,95.8,0.018673,4.0
400,67,Stability,86.2,0.019062,3.0
483,81,Planning,91.2,0.019615,4.0
596,100,IBR,96.4,0.020222,4.0
598,100,Stability,87.2,0.024862,5.0
502,84,Stability,93.0,0.026339,6.0


In [26]:
prp_coherency

,Abstract_Index,PRP_Name,mean_affinity,std_affinity,min_affinity,max_affinity,n_models,mad_affinity,cv,range,Affinity_Level,cv_low_mean_flag,singleton_flag
0,1,CROF,24.0,14.317821,5.0,40.0,5,11.20,0.596576,35.0,Low,False,False
1,1,DER,19.0,6.519202,10.0,25.0,5,5.20,0.343116,15.0,Low,False,False
2,1,IBR,17.0,9.746794,5.0,30.0,5,7.60,0.573341,25.0,Low,False,False
3,1,Planning,79.6,11.760102,60.0,90.0,5,8.48,0.147740,30.0,High,False,False
4,1,Stability,18.0,7.582875,10.0,30.0,5,5.60,0.421271,20.0,Low,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
595,100,DER,22.6,6.985700,18.0,35.0,5,4.96,0.309102,17.0,Low,False,False
596,100,IBR,96.4,1.949359,95.0,99.0,5,1.68,0.020222,4.0,High,False,False
597,100,Planning,12.4,2.509980,10.0,15.0,5,2.08,0.202418,5.0,Low,False,False
598,100,Stability,87.2,2.167948,85.0,90.0,5,1.76,0.024862,5.0,High,False,False


In [27]:
fig1 = px.box(
    prp_df[prp_df['Abstract_Index'].isin(prp_coherency.nlargest(7, 'cv')['Abstract_Index'])],
    y='LLM_Affinity',
    points='all',
    facet_col='Abstract_Index',
    color = 'PRP_Name',
    title=f'PRP Affinity Distribution (top CV abstracts)'
 )
fig2 = px.box(
    prp_df[prp_df['Abstract_Index'].isin(prp_coherency.nsmallest(5, 'cv')['Abstract_Index'])],
    y='LLM_Affinity',
    points='all',
    facet_col='Abstract_Index',
    color = 'PRP_Name',
    title=f'PRP Affinity Distribution (low CV abstracts)'
 )
# fig.update_xaxes(range=[0.5,10.5])
fig1.update_layout(yaxis_title='LLM Affinity')
fig2.update_layout(yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig1.show()
fig2.show()

In [28]:
ra_coherency = compute_unit_coherency(ra_df, target_col='RA2025_ID')

print("RA Coherency Summary:")
print(f"Mean CV across all abstract-RA pairs: {ra_coherency['cv'].mean():.3f}")
print(f"Abstracts with perfect agreement (CV=0): {(ra_coherency['cv'] == 0).sum()}")
print("\nBottom 10 coherent (high CV) abstract-RA pairs:")
display(ra_coherency.nlargest(10, 'cv')[['Abstract_Index', 'RA2025_ID', 'mean_affinity', 'cv', 'range']])
print("\nTop 10 coherent (low CV) abstract-RA pairs:")
display(ra_coherency.nsmallest(10, 'cv')[['Abstract_Index', 'RA2025_ID', 'mean_affinity', 'cv', 'range']])

NameError: name 'ra_df' is not defined

In [ ]:
# Abstract-level coherency: aggregate CV per abstract (for both PRP and RA)
prp_abstract_coherency = (
    prp_coherency.groupby('Abstract_Index', as_index=False)['cv']
    .agg(mean_cv='mean', max_cv='max', min_cv='min', std_cv='std', n_pairs='count')
    .sort_values('mean_cv', ascending=True)
)

ra_abstract_coherency = (
    ra_coherency.groupby('Abstract_Index', as_index=False)['cv']
    .agg(mean_cv='mean', max_cv='max', min_cv='min', std_cv='std', n_pairs='count')
    .sort_values('mean_cv', ascending=True)
)

print("PRP Abstract-level Coherency (Average CV per Abstract):")
print(f"Most coherent abstracts (lowest mean CV):")
display(prp_abstract_coherency.head(10))
print(f"\nLeast coherent abstracts (highest mean CV):")
display(prp_abstract_coherency.tail(10))

print("\n" + "="*60)
print("\nRA Abstract-level Coherency (Average CV per Abstract):")
print(f"Most coherent abstracts (lowest mean CV):")
display(ra_abstract_coherency.head(10))
print(f"\nLeast coherent abstracts (highest mean CV):")
display(ra_abstract_coherency.tail(10))

PRP Abstract-level Coherency (Average CV per Abstract):
Most coherent abstracts (lowest mean CV):


,Abstract_Index,mean_cv,max_cv,min_cv,std_cv,n_pairs
18,19,0.272370,0.424068,0.068646,0.124351,6
43,44,0.425734,0.851591,0.078437,0.293063,6
17,18,0.443337,0.628616,0.130547,0.179574,6
19,20,0.446908,0.815892,0.117635,0.312129,6
14,15,0.473802,0.813680,0.219177,0.200101,6
47,48,0.487941,0.746648,0.230228,0.195083,6
41,42,0.506209,0.817675,0.142117,0.249991,6
21,22,0.532893,0.748418,0.149002,0.223952,6
23,24,0.556627,0.973911,0.250667,0.298825,6
35,36,0.562114,0.889370,0.266259,0.250054,6



Least coherent abstracts (highest mean CV):


,Abstract_Index,mean_cv,max_cv,min_cv,std_cv,n_pairs
30,31,0.732165,1.030958,0.506115,0.170253,6
5,6,0.741391,1.285183,0.443480,0.296638,6
1,2,0.748457,1.073429,0.085018,0.341229,6
42,43,0.802489,1.119989,0.377956,0.287348,6
6,7,0.850932,1.123779,0.621990,0.181490,6
46,47,0.851813,1.093583,0.479489,0.236737,6
33,34,0.858460,1.351898,0.549304,0.276899,6
2,3,0.913666,1.106309,0.337714,0.288081,6
34,35,0.969880,1.352342,0.383793,0.399606,6
10,11,1.076625,1.819318,0.825259,0.368302,6




RA Abstract-level Coherency (Average CV per Abstract):
Most coherent abstracts (lowest mean CV):


,Abstract_Index,mean_cv,max_cv,min_cv,std_cv,n_pairs
16,17,0.224970,0.415945,0.030744,0.138158,6
18,19,0.238108,0.642824,0.074432,0.165159,12
11,12,0.257920,0.866025,0.000000,0.237353,12
0,1,0.279922,0.666667,0.062782,0.238415,7
49,50,0.296829,0.400302,0.176777,0.082977,6
24,25,0.337321,0.534363,0.181818,0.153286,6
3,4,0.365846,0.875466,0.000000,0.277159,12
37,38,0.374204,0.773640,0.000000,0.203276,12
45,46,0.389557,0.532908,0.181303,0.120910,6
9,10,0.389831,0.601962,0.074277,0.243502,6



Least coherent abstracts (highest mean CV):


,Abstract_Index,mean_cv,max_cv,min_cv,std_cv,n_pairs
2,3,0.710110,1.092599,0.360070,0.236004,6
29,30,0.736216,1.414214,0.346410,0.324274,13
41,42,0.764783,1.414214,0.000000,0.444586,17
15,16,0.825011,1.060660,0.471405,0.145860,12
47,48,0.833156,1.414214,0.272166,0.559862,11
36,37,0.850589,1.249000,0.606092,0.230828,7
48,49,0.851956,1.414214,0.000000,0.442321,12
28,29,0.971006,1.414214,0.108786,0.573846,12
33,34,1.006043,1.414214,0.101885,0.511274,13
10,11,1.414214,1.414214,1.414214,0.000000,6


In [ ]:
# Visualization: Coherency Distribution
fig = px.histogram(
    prp_coherency[prp_coherency['cv'].notna()],
    x='cv',
    # nbins=30,
    # facet_col = 'PRP_Name',
    title='PRP Coherency Distribution (Coefficient of Variation)',
    labels={'cv': 'Coefficient of Variation', 'count': 'Number of Pairs'}
)
fig.add_vline(x=prp_coherency['cv'].mean(), line_dash="dash", line_color="red", 
              annotation_text=f"Mean CV: {prp_coherency['cv'].mean():.3f}")
fig.show()

fig = px.histogram(
    ra_coherency[ra_coherency['cv'].notna()],
    x='cv',
    nbins=30,
    title='RA Coherency Distribution (Coefficient of Variation)',
    labels={'cv': 'Coefficient of Variation', 'count': 'Number of Pairs'}
)
fig.add_vline(x=ra_coherency['cv'].mean(), line_dash="dash", line_color="red",
              annotation_text=f"Mean CV: {ra_coherency['cv'].mean():.3f}")
fig.show()

In [ ]:
prp_coherency

,Abstract_Index,PRP_Name,mean_affinity,std_affinity,min_affinity,max_affinity,n_models,mad_affinity,cv,range,Affinity_Level,cv_low_mean_flag,singleton_flag
0,1,CROF,19.777778,20.747155,0.0,60.0,9,15.703704,1.049013,60.0,Low,False,False
1,1,DER,33.555556,27.180467,0.0,90.0,9,19.851852,0.810014,90.0,Low,False,False
2,1,IBR,18.666667,18.159020,0.0,45.0,9,15.629630,0.972805,45.0,Low,False,False
3,1,Planning,66.888889,25.241060,20.0,92.0,9,18.617284,0.377358,72.0,Moderate,False,False
4,1,Stability,39.666667,21.644861,10.0,75.0,9,17.037037,0.545669,65.0,Low,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,50,DER,66.000000,17.327723,40.0,90.0,9,14.444444,0.262541,50.0,Moderate,False,False
296,50,IBR,22.555556,22.556103,0.0,75.0,9,14.962963,1.000024,75.0,Low,False,False
297,50,Planning,37.000000,21.118712,5.0,70.0,9,16.222222,0.570776,65.0,Low,False,False
298,50,Stability,40.777778,18.853234,20.0,80.0,9,12.987654,0.462341,60.0,Moderate,False,False


In [ ]:
# Abstract-level coherency scatter plots
fig = px.box(
    prp_coherency,
    x='Abstract_Index',
    y='cv',
    title='PRP Abstract-level Coherency (Lower CV = Higher Agreement)',
    points ='all',
)
fig.update_traces(boxmean=True)  # set to 'sd' to show mean + std
fig.show()

fig = px.box(
    ra_coherency,
    x='Abstract_Index',
    y='cv',
    title='RA Abstract-level Coherency (Lower CV = Higher Agreement)',
)
fig.update_traces(boxmean=True)  # set to 'sd' to show mean + std
fig.show()


In [ ]:
# Abstract-level coherency scatter plots
fig = px.scatter(
    prp_abstract_coherency,
    x='Abstract_Index',
    y='mean_cv',
    size='n_pairs',
    color='mean_cv',
    hover_data=['max_cv', 'std_cv'],
    title='PRP Abstract-level Coherency (Lower CV = Higher Agreement)',
    labels={'mean_cv': 'Mean CV (Coherency)', 'Abstract_Index': 'Abstract Index'},
    color_continuous_scale='RdYlGn_r'
)
fig.show()

fig = px.scatter(
    ra_abstract_coherency,
    x='Abstract_Index',
    y='mean_cv',
    size='n_pairs',
    color='mean_cv',
    hover_data=['max_cv', 'std_cv'],
    title='RA Abstract-level Coherency (Lower CV = Higher Agreement)',
    labels={'mean_cv': 'Mean CV (Coherency)', 'Abstract_Index': 'Abstract Index'},
    color_continuous_scale='RdYlGn_r'
)
fig.show()

In [ ]:
# Pairwise Model Coherency: Correlation between model pairs per abstract-PRP
# This shows which model pairs agree most on affinity levels

from itertools import combinations

# For PRP: compute correlation between each pair of models per abstract
prp_model_pairs = []
for abstract_idx in prp_df['Abstract_Index'].unique():
    for prp_name in prp_df[prp_df['Abstract_Index'] == abstract_idx]['PRP_Name'].unique():
        subset = prp_df[(prp_df['Abstract_Index'] == abstract_idx) & (prp_df['PRP_Name'] == prp_name)]
        
        if len(subset) >= 2:
            models = subset['Model_Slug'].unique()
            for model1, model2 in combinations(models, 2):
                val1 = subset[subset['Model_Slug'] == model1]['LLM_Affinity'].values
                val2 = subset[subset['Model_Slug'] == model2]['LLM_Affinity'].values
                
                if len(val1) > 0 and len(val2) > 0:
                    diff = abs(val1[0] - val2[0])
                    prp_model_pairs.append({
                        'Abstract_Index': abstract_idx,
                        'PRP_Name': prp_name,
                        'Model_1': min(model1, model2),
                        'Model_2': max(model1, model2),
                        'Affinity_Difference': diff
                    })

prp_pairwise = pd.DataFrame(prp_model_pairs)

if len(prp_pairwise) > 0:
    print("PRP Model Pair Agreement Analysis:")
    print("\nModel pairs with highest average disagreement (highest mean affinity difference):")
    pair_stats = prp_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].agg(
        mean_diff='mean', std_diff='std', n_pairs='count'
    ).sort_values('mean_diff', ascending=False)
    display(pair_stats.head(10))
    
    print("\nModel pairs with lowest average disagreement (highest mean agreement):")
    display(pair_stats.sort_values('mean_diff', ascending=True).head(10))

PRP Model Pair Agreement Analysis:

Model pairs with highest average disagreement (highest mean affinity difference):


,Model_1,Model_2,mean_diff,std_diff,n_pairs
18,gpt-5.4-nano,mistral-7b,38.456667,22.353382,300
33,mistral-7b,mixtral-8x7b,31.343537,20.430044,294
12,gpt-5.4-mini,mistral-7b,31.196667,21.270939,300
34,mistral-7b,qwen2.5-7b,31.156667,19.930283,300
17,gpt-5.4-nano,ministral-3-14b-cloud,29.610000,21.926555,300
23,llama3.1-8b,mistral-7b,28.333333,16.652725,300
31,ministral-3-14b-cloud,mixtral-8x7b,27.840136,23.619097,294
5,gemma4-31b-cloud,mistral-7b,27.443333,21.584402,300
7,gemma4-31b-cloud,qwen2.5-7b,26.966667,20.283910,300
25,llama3.1-8b,qwen2.5-7b,26.723333,23.093386,300



Model pairs with lowest average disagreement (highest mean agreement):


,Model_1,Model_2,mean_diff,std_diff,n_pairs
0,gemma4-31b-cloud,gpt-5.4-mini,13.546667,12.992710,300
8,gpt-5.4-mini,gpt-5.4-nano,17.713333,17.312526,300
21,llama3.1-8b,llama4-16x17b,19.266667,13.045927,300
27,llama4-16x17b,mistral-7b,19.333333,14.523268,300
26,llama4-16x17b,ministral-3-14b-cloud,20.000000,14.896522,300
11,gpt-5.4-mini,ministral-3-14b-cloud,20.050000,17.805363,300
19,gpt-5.4-nano,mixtral-8x7b,20.153061,18.264282,294
4,gemma4-31b-cloud,ministral-3-14b-cloud,20.263333,17.826758,300
24,llama3.1-8b,mixtral-8x7b,20.578231,20.831823,294
28,llama4-16x17b,mixtral-8x7b,22.125850,14.522707,294


In [ ]:
# For RA: compute correlation between each pair of models per abstract
ra_model_pairs = []
for abstract_idx in ra_df['Abstract_Index'].unique():
    for ra_id in ra_df[ra_df['Abstract_Index'] == abstract_idx]['RA2025_ID'].unique():
        subset = ra_df[(ra_df['Abstract_Index'] == abstract_idx) & (ra_df['RA2025_ID'] == ra_id)]
        
        if len(subset) >= 2:
            models = subset['Model_Slug'].unique()
            for model1, model2 in combinations(models, 2):
                val1 = subset[subset['Model_Slug'] == model1]['LLM_Affinity'].values
                val2 = subset[subset['Model_Slug'] == model2]['LLM_Affinity'].values
                
                if len(val1) > 0 and len(val2) > 0:
                    diff = abs(val1[0] - val2[0])
                    ra_model_pairs.append({
                        'Abstract_Index': abstract_idx,
                        'RA2025_ID': ra_id,
                        'Model_1': min(model1, model2),
                        'Model_2': max(model1, model2),
                        'Affinity_Difference': diff
                    })

ra_pairwise = pd.DataFrame(ra_model_pairs)

if len(ra_pairwise) > 0:
    print("RA Model Pair Agreement Analysis:")
    print("\nModel pairs with highest average disagreement (highest mean affinity difference):")
    pair_stats_ra = ra_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].agg(
        mean_diff='mean', std_diff='std', n_pairs='count'
    ).sort_values('mean_diff', ascending=False)
    display(pair_stats_ra.head(10))
    
    print("\nModel pairs with lowest average disagreement (highest mean agreement):")
    display(pair_stats_ra.sort_values('mean_diff', ascending=True).head(10))

RA Model Pair Agreement Analysis:

Model pairs with highest average disagreement (highest mean affinity difference):


,Model_1,Model_2,mean_diff,std_diff,n_pairs
8,llama3.1-8b,mistral-7b,39.809524,23.796258,21
19,mistral-7b,qwen2.5-7b,37.540000,20.658753,50
12,llama4-16x17b,mistral-7b,31.440000,23.570957,25
9,llama3.1-8b,mixtral-8x7b,30.673077,18.258193,52
7,llama3.1-8b,ministral-3-14b-cloud,30.620155,20.039100,129
10,llama3.1-8b,qwen2.5-7b,26.564103,18.525931,78
18,mistral-7b,mixtral-8x7b,24.640000,17.920844,25
17,ministral-3-14b-cloud,qwen2.5-7b,24.430657,17.993788,137
6,llama3.1-8b,llama4-16x17b,23.551136,14.862328,176
0,gemma4-31b-cloud,llama3.1-8b,22.391304,20.408398,138



Model pairs with lowest average disagreement (highest mean agreement):


,Model_1,Model_2,mean_diff,std_diff,n_pairs
20,mixtral-8x7b,qwen2.5-7b,12.351852,9.462484,108
1,gemma4-31b-cloud,llama4-16x17b,14.897959,14.111462,147
15,ministral-3-14b-cloud,mistral-7b,15.391304,12.608580,23
2,gemma4-31b-cloud,ministral-3-14b-cloud,15.728477,13.945099,151
4,gemma4-31b-cloud,mixtral-8x7b,16.238095,11.803295,105
14,llama4-16x17b,qwen2.5-7b,16.326316,11.349798,95
13,llama4-16x17b,mixtral-8x7b,16.917808,12.545200,73
11,llama4-16x17b,ministral-3-14b-cloud,19.720000,16.017933,125
5,gemma4-31b-cloud,qwen2.5-7b,19.794118,15.552595,136
3,gemma4-31b-cloud,mistral-7b,19.861111,14.710837,36


In [ ]:
# Model Agreement Heatmaps: Show which models have smallest disagreements
if len(prp_pairwise) > 0:
    pair_stats = prp_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].agg(
        mean_diff='mean', n_pairs='count'
    ).sort_values('mean_diff', ascending=True)
    
    # Create a symmetrical matrix for heatmap
    all_models = sorted(set(list(pair_stats['Model_1'].unique()) + list(pair_stats['Model_2'].unique())))
    model_agreement_matrix = pd.DataFrame(0.0, index=all_models, columns=all_models)
    
    for _, row in pair_stats.iterrows():
        model_agreement_matrix.loc[row['Model_1'], row['Model_2']] = row['mean_diff']
        model_agreement_matrix.loc[row['Model_2'], row['Model_1']] = row['mean_diff']
    
    fig = px.imshow(
        model_agreement_matrix,
        text_auto='.2f',
        color_continuous_scale='RdYlGn_r',
        title='PRP Model Pairwise Disagreement (Lower = Better Agreement)',
        labels={'color': 'Mean Affinity Difference'},
        aspect='auto'
    )
    fig.update_layout(xaxis_title='Model', yaxis_title='Model')
    fig.show()

if len(ra_pairwise) > 0:
    pair_stats_ra = ra_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].agg(
        mean_diff='mean', n_pairs='count'
    ).sort_values('mean_diff', ascending=True)
    
    all_models_ra = sorted(set(list(pair_stats_ra['Model_1'].unique()) + list(pair_stats_ra['Model_2'].unique())))
    model_agreement_matrix_ra = pd.DataFrame(0.0, index=all_models_ra, columns=all_models_ra)
    
    for _, row in pair_stats_ra.iterrows():
        model_agreement_matrix_ra.loc[row['Model_1'], row['Model_2']] = row['mean_diff']
        model_agreement_matrix_ra.loc[row['Model_2'], row['Model_1']] = row['mean_diff']
    
    fig = px.imshow(
        model_agreement_matrix_ra,
        text_auto='.2f',
        color_continuous_scale='RdYlGn_r',
        title='RA Model Pairwise Disagreement (Lower = Better Agreement)',
        labels={'color': 'Mean Affinity Difference'},
        aspect='auto'
    )
    fig.update_layout(xaxis_title='Model', yaxis_title='Model')
    fig.show()

## Coherency Key Insights

Summary of model agreement levels on affinity assignments.

In [ ]:
# Summary statistics on model coherency
print("=" * 70)
print("COHERENCY ANALYSIS SUMMARY")
print("=" * 70)

print("\n📊 PRP COHERENCY:")
print(f"  • Mean CV (lower = more agreement): {prp_coherency['cv'].mean():.3f}")
print(f"  • Median CV: {prp_coherency['cv'].median():.3f}")
print(f"  • Perfect agreement cases (CV=0): {(prp_coherency['cv'] == 0).sum()} / {len(prp_coherency)}")
print(f"  • High disagreement cases (CV>1): {(prp_coherency['cv'] > 1).sum()} / {len(prp_coherency)}")
print(f"  • Range of affinities (mean): {prp_coherency['range'].mean():.3f}")
print(f"  • Most coherent abstract: {prp_abstract_coherency.iloc[0]['Abstract_Index']} (mean CV: {prp_abstract_coherency.iloc[0]['mean_cv']:.3f})")
print(f"  • Least coherent abstract: {prp_abstract_coherency.iloc[-1]['Abstract_Index']} (mean CV: {prp_abstract_coherency.iloc[-1]['mean_cv']:.3f})")

print("\n📊 RA COHERENCY:")
print(f"  • Mean CV (lower = more agreement): {ra_coherency['cv'].mean():.3f}")
print(f"  • Median CV: {ra_coherency['cv'].median():.3f}")
print(f"  • Perfect agreement cases (CV=0): {(ra_coherency['cv'] == 0).sum()} / {len(ra_coherency)}")
print(f"  • High disagreement cases (CV>1): {(ra_coherency['cv'] > 1).sum()} / {len(ra_coherency)}")
print(f"  • Range of affinities (mean): {ra_coherency['range'].mean():.3f}")
print(f"  • Most coherent abstract: {ra_abstract_coherency.iloc[0]['Abstract_Index']} (mean CV: {ra_abstract_coherency.iloc[0]['mean_cv']:.3f})")
print(f"  • Least coherent abstract: {ra_abstract_coherency.iloc[-1]['Abstract_Index']} (mean CV: {ra_abstract_coherency.iloc[-1]['mean_cv']:.3f})")

if len(prp_pairwise) > 0:
    pair_stats = prp_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].mean()
    print("\n🔗 PRP MODEL PAIRS - BEST AGREEMENT (Lowest Mean Difference):")
    for i, (_, row) in enumerate(pair_stats.sort_values('Affinity_Difference').head(3).iterrows(), 1):
        print(f"  {i}. {row['Model_1']} ↔ {row['Model_2']}: {row['Affinity_Difference']:.3f}")

if len(ra_pairwise) > 0:
    pair_stats_ra = ra_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].mean()
    print("\n🔗 RA MODEL PAIRS - BEST AGREEMENT (Lowest Mean Difference):")
    for i, (_, row) in enumerate(pair_stats_ra.sort_values('Affinity_Difference').head(3).iterrows(), 1):
        print(f"  {i}. {row['Model_1']} ↔ {row['Model_2']}: {row['Affinity_Difference']:.3f}")

print("\n" + "=" * 70)

COHERENCY ANALYSIS SUMMARY

📊 PRP COHERENCY:
  • Mean CV (lower = more agreement): 0.654
  • Median CV: 0.647
  • Perfect agreement cases (CV=0): 0 / 300
  • High disagreement cases (CV>1): 27 / 300
  • Range of affinities (mean): 61.513
  • Most coherent abstract: 19.0 (mean CV: 0.272)
  • Least coherent abstract: 11.0 (mean CV: 1.077)

📊 RA COHERENCY:
  • Mean CV (lower = more agreement): 0.564
  • Median CV: 0.471
  • Perfect agreement cases (CV=0): 14 / 897
  • High disagreement cases (CV>1): 60 / 897
  • Range of affinities (mean): 18.093
  • Most coherent abstract: 17.0 (mean CV: 0.225)
  • Least coherent abstract: 11.0 (mean CV: 1.414)

🔗 PRP MODEL PAIRS - BEST AGREEMENT (Lowest Mean Difference):
  1. gemma4-31b-cloud ↔ gpt-5.4-mini: 13.547
  2. gpt-5.4-mini ↔ gpt-5.4-nano: 17.713
  3. llama3.1-8b ↔ llama4-16x17b: 19.267

🔗 RA MODEL PAIRS - BEST AGREEMENT (Lowest Mean Difference):
  1. mixtral-8x7b ↔ qwen2.5-7b: 12.352
  2. gemma4-31b-cloud ↔ llama4-16x17b: 14.898
  3. ministral

## Coherency by Fixed Affinity Levels (CV + MAD)

This section extends coherency analysis with fixed affinity bands and a robust companion metric:

- Low: 0-40
- Moderate: 41-70
- High: 71-100

Metrics per abstract-target unit across models:
- CV = std / mean (with epsilon)
- MAD = mean absolute deviation from cross-model mean

CV can be unstable when mean affinity is very low, so a low-mean caution flag is included.

In [ ]:
# PRP coherency by fixed levels
prp_unit_coherency_lvl = compute_unit_coherency(prp_df, target_col='PRP_Name')
prp_level_summary = summarize_coherency_by_level(prp_unit_coherency_lvl)

print('PRP level-wise coherency summary (CV + MAD):')
display(prp_level_summary)

print('\nMost incoherent PRP units by CV:')
display(
    prp_unit_coherency_lvl.sort_values('cv', ascending=False)[
        ['Abstract_Index', 'PRP_Name', 'mean_affinity', 'cv', 'mad_affinity', 'range', 'n_models', 'cv_low_mean_flag']
    ].head(15)
)

print('\nMost incoherent PRP units by MAD:')
display(
    prp_unit_coherency_lvl.sort_values('mad_affinity', ascending=False)[
        ['Abstract_Index', 'PRP_Name', 'mean_affinity', 'cv', 'mad_affinity', 'range', 'n_models', 'cv_low_mean_flag']
    ].head(15)
)

fig = px.box(
    prp_unit_coherency_lvl.dropna(subset=['Affinity_Level']),
    x='Affinity_Level',
    y='cv',
    color='Affinity_Level',
    category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
    title='PRP Coherency by Affinity Level (CV)',
    points='all',
)
fig.update_layout(xaxis_title='Affinity level', yaxis_title='CV across models')
fig.show()

fig = px.box(
    prp_unit_coherency_lvl.dropna(subset=['Affinity_Level']),
    x='Affinity_Level',
    y='mad_affinity',
    color='Affinity_Level',
    category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
    title='PRP Coherency by Affinity Level (MAD)',
    points='all',
)
fig.update_layout(xaxis_title='Affinity level', yaxis_title='MAD across models')
fig.show()

PRP level-wise coherency summary (CV + MAD):


,Affinity_Level,n_units,mean_cv,median_cv,p90_cv,mean_mad,median_mad,mean_range,pct_low_mean_flag,pct_singleton
0,Low,180,0.812628,0.809475,1.056677,16.226668,15.851852,59.766667,0.0,0.0
1,Moderate,95,0.476770,0.479489,0.635212,19.409747,19.234568,70.052632,0.0,0.0
2,High,25,0.181647,0.176728,0.291720,11.005432,10.444444,41.640000,0.0,0.0



Most incoherent PRP units by CV:


,Abstract_Index,PRP_Name,mean_affinity,cv,mad_affinity,range,n_models,cv_low_mean_flag
62,11,IBR,6.111111,1.819318,8.395062,30.0,9,False
209,35,System Services,19.750000,1.352342,21.437500,70.0,8,False
200,34,IBR,8.555556,1.351898,9.851852,30.0,9,False
206,35,IBR,10.875000,1.338363,11.843750,40.0,8,False
32,6,IBR,9.111111,1.285183,10.123457,30.0,9,False
205,35,DER,23.500000,1.183648,21.500000,80.0,8,False
50,9,IBR,27.222222,1.147449,26.913580,75.0,9,False
38,7,IBR,14.222222,1.123779,14.024691,40.0,9,False
254,43,IBR,11.111111,1.119989,10.370370,30.0,9,False
13,3,DER,20.666667,1.106309,17.185185,70.0,9,False



Most incoherent PRP units by MAD:


,Abstract_Index,PRP_Name,mean_affinity,cv,mad_affinity,range,n_models,cv_low_mean_flag
248,42,IBR,51.000000,0.725225,31.111111,92.0,9,False
140,24,IBR,37.000000,0.973911,30.000000,90.0,9,False
146,25,IBR,42.555556,0.814452,29.506173,85.0,9,False
92,16,IBR,47.000000,0.714748,28.444444,85.0,9,False
130,22,Stability,51.888889,0.610684,28.345679,74.0,9,False
98,17,IBR,46.444444,0.734910,28.271605,90.0,9,False
39,7,Planning,55.444444,0.621990,27.061728,90.0,9,False
50,9,IBR,27.222222,1.147449,26.913580,75.0,9,False
9,2,Planning,39.777778,0.777074,26.864198,80.0,9,False
53,9,System Services,53.111111,0.568252,26.543210,72.0,9,False


In [ ]:
# RA coherency by fixed levels
ra_unit_coherency_lvl = compute_unit_coherency(ra_df, target_col='RA2025_ID')
ra_level_summary = summarize_coherency_by_level(ra_unit_coherency_lvl)

print('RA level-wise coherency summary (CV + MAD):')
display(ra_level_summary)

print('\nMost incoherent RA units by CV:')
display(
    ra_unit_coherency_lvl.sort_values('cv', ascending=False)[
        ['Abstract_Index', 'RA2025_ID', 'mean_affinity', 'cv', 'mad_affinity', 'range', 'n_models', 'cv_low_mean_flag']
    ].head(15)
)

print('\nMost incoherent RA units by MAD:')
display(
    ra_unit_coherency_lvl.sort_values('mad_affinity', ascending=False)[
        ['Abstract_Index', 'RA2025_ID', 'mean_affinity', 'cv', 'mad_affinity', 'range', 'n_models', 'cv_low_mean_flag']
    ].head(15)
)

fig = px.box(
    ra_unit_coherency_lvl.dropna(subset=['Affinity_Level']),
    x='Affinity_Level',
    y='cv',
    color='Affinity_Level',
    category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
    title='RA Coherency by Affinity Level (CV)'
)
fig.update_layout(xaxis_title='Affinity level', yaxis_title='CV across models')
fig.show()

fig = px.box(
    ra_unit_coherency_lvl.dropna(subset=['Affinity_Level']),
    x='Affinity_Level',
    y='mad_affinity',
    color='Affinity_Level',
    category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
    title='RA Coherency by Affinity Level (MAD)'
)
fig.update_layout(xaxis_title='Affinity level', yaxis_title='MAD across models')
fig.show()

RA level-wise coherency summary (CV + MAD):


,Affinity_Level,n_units,mean_cv,median_cv,p90_cv,mean_mad,median_mad,mean_range,pct_low_mean_flag,pct_singleton
0,Low,544,0.666273,0.606092,1.414214,6.230989,5.0,15.461397,4.963235,44.852941
1,Moderate,208,0.407745,0.396904,0.641678,10.336117,10.0,27.475962,0.000000,31.730769
2,High,55,0.147854,0.124042,0.324766,3.055273,0.0,8.636364,0.000000,63.636364



Most incoherent RA units by CV:


,Abstract_Index,RA2025_ID,mean_affinity,cv,mad_affinity,range,n_models,cv_low_mean_flag
230,13,37,45.0,1.414214,45.0,90.0,2,False
505,30,17,37.5,1.414214,37.5,75.0,2,False
847,48,10,37.5,1.414214,37.5,75.0,2,False
583,34,17,37.5,1.414214,37.5,75.0,2,False
476,29,1,32.5,1.414214,32.5,65.0,2,False
844,48,7,22.5,1.414214,22.5,45.0,2,False
588,34,32,20.0,1.414214,20.0,40.0,2,False
589,34,33,20.0,1.414214,20.0,40.0,2,False
846,48,9,17.5,1.414214,17.5,35.0,2,False
758,42,14,15.0,1.414214,15.0,30.0,2,False



Most incoherent RA units by MAD:


,Abstract_Index,RA2025_ID,mean_affinity,cv,mad_affinity,range,n_models,cv_low_mean_flag
230,13,37,45.000000,1.414214,45.000000,90.0,2,False
583,34,17,37.500000,1.414214,37.500000,75.0,2,False
847,48,10,37.500000,1.414214,37.500000,75.0,2,False
505,30,17,37.500000,1.414214,37.500000,75.0,2,False
695,39,28,55.000000,0.899954,35.000000,70.0,2,False
133,8,37,35.000000,1.269742,33.333333,85.0,3,False
57,4,31,52.500000,0.875466,32.500000,65.0,2,False
476,29,1,32.500000,1.414214,32.500000,65.0,2,False
860,49,17,43.333333,1.040596,31.111111,90.0,3,False
748,42,4,50.000000,0.848528,30.000000,60.0,2,False


In [ ]:
# Pairwise model disagreement by level (PRP)
prp_pairwise_lvl, prp_pairwise_lvl_summary = compute_pairwise_disagreement_by_level(prp_df, target_col='PRP_Name')

if not prp_pairwise_lvl_summary.empty:
    print('PRP pairwise disagreement by level (highest mean difference):')
    display(prp_pairwise_lvl_summary.head(20))

    for level in ['Low', 'Moderate', 'High']:
        level_df = prp_pairwise_lvl_summary[prp_pairwise_lvl_summary['Affinity_Level'] == level]
        if level_df.empty:
            continue

        models = sorted(set(level_df['Model_1']).union(level_df['Model_2']))
        mat = pd.DataFrame(np.nan, index=models, columns=models)
        np.fill_diagonal(mat.values, 0.0)

        for _, row in level_df.iterrows():
            mat.loc[row['Model_1'], row['Model_2']] = row['mean_diff']
            mat.loc[row['Model_2'], row['Model_1']] = row['mean_diff']

        fig = px.imshow(
            mat,
            text_auto='.2f',
            color_continuous_scale='RdYlGn_r',
            title=f'PRP Pairwise Disagreement ({level})',
            labels={'color': 'Mean absolute difference'},
            aspect='auto'
        )
        fig.update_layout(xaxis_title='Model', yaxis_title='Model')
        fig.show()
else:
    print('No PRP pairwise level summary available.')

PRP pairwise disagreement by level (highest mean difference):


,Affinity_Level,Model_1,Model_2,mean_diff,median_diff,std_diff,n_pairs
25,High,llama3.1-8b,qwen2.5-7b,23.00,15.0,21.262251,25
10,High,gpt-5.4-mini,llama4-16x17b,21.36,18.0,10.617909,25
14,High,gpt-5.4-mini,qwen2.5-7b,21.20,15.0,20.822664,25
20,High,gpt-5.4-nano,qwen2.5-7b,20.96,18.0,18.762729,25
13,High,gpt-5.4-mini,mixtral-8x7b,20.12,15.0,19.511364,25
3,High,gemma4-31b-cloud,llama4-16x17b,19.68,15.0,10.995150,25
7,High,gemma4-31b-cloud,qwen2.5-7b,19.68,15.0,20.032723,25
29,High,llama4-16x17b,qwen2.5-7b,19.20,10.0,15.253415,25
9,High,gpt-5.4-mini,llama3.1-8b,18.76,8.0,18.083786,25
6,High,gemma4-31b-cloud,mixtral-8x7b,18.68,12.0,19.034442,25


In [ ]:
# Pairwise model disagreement by level (RA)
ra_pairwise_lvl, ra_pairwise_lvl_summary = compute_pairwise_disagreement_by_level(ra_df, target_col='RA2025_ID')

if not ra_pairwise_lvl_summary.empty:
    print('RA pairwise disagreement by level (highest mean difference):')
    display(ra_pairwise_lvl_summary.head(20))

    for level in ['Low', 'Moderate', 'High']:
        level_df = ra_pairwise_lvl_summary[ra_pairwise_lvl_summary['Affinity_Level'] == level]
        if level_df.empty:
            continue

        models = sorted(set(level_df['Model_1']).union(level_df['Model_2']))
        mat = pd.DataFrame(np.nan, index=models, columns=models)
        np.fill_diagonal(mat.values, 0.0)

        for _, row in level_df.iterrows():
            mat.loc[row['Model_1'], row['Model_2']] = row['mean_diff']
            mat.loc[row['Model_2'], row['Model_1']] = row['mean_diff']

        fig = px.imshow(
            mat,
            text_auto='.2f',
            color_continuous_scale='RdYlGn_r',
            title=f'RA Pairwise Disagreement ({level})',
            labels={'color': 'Mean absolute difference'},
            aspect='auto'
        )
        fig.update_layout(xaxis_title='Model', yaxis_title='Model')
        fig.show()
else:
    print('No RA pairwise level summary available.')

RA pairwise disagreement by level (highest mean difference):


,Affinity_Level,Model_1,Model_2,mean_diff,median_diff,std_diff,n_pairs
17,High,mistral-7b,qwen2.5-7b,45.000000,45.0,NaN,1
4,High,gemma4-31b-cloud,mixtral-8x7b,35.000000,35.0,NaN,1
15,High,ministral-3-14b-cloud,qwen2.5-7b,30.285714,19.0,25.031409,7
9,High,llama3.1-8b,qwen2.5-7b,27.166667,17.0,28.568631,6
14,High,ministral-3-14b-cloud,mixtral-8x7b,25.000000,25.0,14.142136,2
12,High,llama4-16x17b,qwen2.5-7b,22.166667,19.5,19.903936,6
16,High,mistral-7b,mixtral-8x7b,20.000000,20.0,NaN,1
7,High,llama3.1-8b,ministral-3-14b-cloud,17.777778,10.0,19.860625,9
13,High,ministral-3-14b-cloud,mistral-7b,17.500000,17.5,3.535534,2
5,High,gemma4-31b-cloud,qwen2.5-7b,17.300000,9.5,20.011386,10


In [ ]:
# Compact interpretation table
interpretation_summary = pd.DataFrame(
    [
        {
            'Dataset': 'PRP',
            'Mean_CV': prp_unit_coherency_lvl['cv'].mean(),
            'Median_CV': prp_unit_coherency_lvl['cv'].median(),
            'Mean_MAD': prp_unit_coherency_lvl['mad_affinity'].mean(),
            'Median_MAD': prp_unit_coherency_lvl['mad_affinity'].median(),
            'Pct_Low_Mean_Flag': 100.0 * prp_unit_coherency_lvl['cv_low_mean_flag'].mean(),
            'N_Units': len(prp_unit_coherency_lvl),
        },
        {
            'Dataset': 'RA',
            'Mean_CV': ra_unit_coherency_lvl['cv'].mean(),
            'Median_CV': ra_unit_coherency_lvl['cv'].median(),
            'Mean_MAD': ra_unit_coherency_lvl['mad_affinity'].mean(),
            'Median_MAD': ra_unit_coherency_lvl['mad_affinity'].median(),
            'Pct_Low_Mean_Flag': 100.0 * ra_unit_coherency_lvl['cv_low_mean_flag'].mean(),
            'N_Units': len(ra_unit_coherency_lvl),
        },
    ]
)

print('Global coherency summary (CV + MAD):')
display(interpretation_summary)

print('Interpretation notes:')
print('- Low CV + Low MAD: strong cross-model agreement')
print('- High CV + Low mean affinity: inspect MAD before concluding disagreement')
print('- High CV + High MAD: substantive disagreement across models')

Global coherency summary (CV + MAD):


,Dataset,Mean_CV,Median_CV,Mean_MAD,Median_MAD,Pct_Low_Mean_Flag,N_Units
0,PRP,0.653691,0.647407,16.799540,16.91358,0.000000,300
1,RA,0.564369,0.471405,7.072628,5.00000,3.010033,897


Interpretation notes:
- Low CV + Low MAD: strong cross-model agreement
- High CV + Low mean affinity: inspect MAD before concluding disagreement
- High CV + High MAD: substantive disagreement across models
